<img src="./img/uva_seal.png">  

## Delta Lake with PySpark
### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: November 17, 2026
---

### SOURCES

https://docs.delta.io/latest/quick-start.html

https://docs.delta.io/latest/delta-batch.html

https://docs.delta.io/latest/delta-update.html

https://github.com/delta-io/delta/blob/master/PROTOCOL.md

Learning Spark (2nd ed.), Chapter 9: Building Reliable Data Lakes with Apache Spark

Delta Lake paper: [data_lakehouse.pdf](data_lakehouse.pdf)

### ABOUT

A data lake is a folder of files (usually Parquet) in cheap storage. That is great for scale and cost, but a folder of files gives you none of the guarantees a database does: a crashed job can leave half-written data, nothing stops a bad schema from being appended, and there is no way to update a row or see yesterday's version.

**Delta Lake** fixes this with one idea: next to the Parquet files, keep an ordered **transaction log** (`_delta_log/`) that records which files make up the table at each version. Readers trust the log, not the folder listing. Everything else (ACID writes, schema enforcement, updates, merges, time travel) follows from that log.

This demo uses **open source Delta Lake with plain PySpark** running locally. No Databricks account or cluster is needed, and every file Delta writes lands in a local folder you can open and inspect.

| Lakehouse feature | Plain Parquet folder | Delta Lake |
|---|---|---|
| Atomic writes | a failed job can leave partial or missing data | a commit either lands in the log or it doesn't |
| Schema enforcement | any schema can be appended | mismatched writes are rejected |
| Schema evolution | manual | `mergeSchema` adds columns in one commit |
| UPDATE / DELETE / MERGE | rewrite the folder yourself | built in |
| Time travel | no | read any past version |
| Small-file compaction | manual | `OPTIMIZE`, `VACUUM` |
| Streaming sink with exactly-once | no | yes, via the log |

### OBJECTIVES
- See the failure modes of a plain Parquet data lake
- Create a Delta table and read its transaction log by hand
- Show that a failed write leaves a Delta table untouched (atomicity)
- Use schema enforcement and schema evolution
- Run UPDATE, conditional UPDATE, DELETE, and MERGE (upsert) with both the Python API and SQL
- Time travel to earlier versions and restore a table
- Compact small files with `OPTIMIZE` / Z-ordering and clean up with `VACUUM`
- Use a Delta table as a streaming source and sink
---

### SETUP

Delta Lake ships as two pieces:

- the `delta-spark` **Python package**, which gives us `from delta.tables import DeltaTable`
- the `io.delta:delta-spark_2.13` **JAR**, which Spark downloads from Maven the first time the session starts

`configure_spark_with_delta_pip()` adds the JAR that matches the installed Python package to `spark.jars.packages`. The two `spark.sql` configs turn on Delta's SQL commands (`UPDATE`, `MERGE`, `VACUUM`, ...) and the Delta catalog.

**Notes**
- Start from a **fresh kernel**. JARs can only be added when the Spark session is first created.
- The Delta version must match your Spark version. Check the [compatibility table](https://docs.delta.io/latest/releases.html) and pin it if needed, e.g. `pip install --no-deps delta-spark==<version>`.
- We install with `--no-deps` so pip doesn't replace the kernel's PySpark with a different version.

In [1]:
try:
    import delta
except ImportError:
    %pip install --quiet --no-deps delta-spark
    import delta

In [2]:
try:
    import importlib.metadata
except ImportError:
    %pip install --quiet --no-deps importlib-metadata
    import importlib.metadata

In [3]:
import os
import json
import glob
import shutil
import importlib.metadata

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (SparkSession.builder
    .appName("DeltaLake-Demo")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    # lets us VACUUM with a 0-hour retention in section 8 (never do this in production)
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    # make UPDATE/DELETE rewrite files rather than write deletion vectors, so the log is easy to follow
    .config("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# forward slashes so the paths also work inside SQL strings on Windows
WORK_DIR = os.path.abspath("delta_workspace").replace("\\", "/")
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR)

print(f"Spark {spark.version}, delta-spark {importlib.metadata.version('delta-spark')}")
print(f"workspace at {WORK_DIR}")

:: loading settings :: url = jar:file:/sfs/gpfs/tardis/applications/202606/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/apt4c/.ivy2.5.2/cache
The jars for the packages stored in: /home/apt4c/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2c6a9974-516b-422b-bcdb-c9bc76f2cc08;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#lo

Spark 4.2.0, delta-spark 4.4.1
workspace at /sfs/gpfs/tardis/home/apt4c/distributed_computing/11_delta_lakes/delta_workspace


A few helpers we use throughout:

- `show_log(path)` prints the actions in each commit file of a table's `_delta_log/`
- `data_files(path)` lists the Parquet files physically sitting in the table folder
- `attempt(fn)` runs a function and prints the error instead of a long stack trace

In [4]:
def show_log(path, versions=None):
    """Print a one-line summary of every action in each commit of a Delta table."""
    for log_file in sorted(glob.glob(f"{path}/_delta_log/*.json")):
        version = int(os.path.basename(log_file).split(".")[0])
        if versions is not None and version not in versions:
            continue
        print(f"--- version {version}: _delta_log/{os.path.basename(log_file)}")
        with open(log_file) as fh:
            for line in fh:
                action = json.loads(line)
                kind, body = next(iter(action.items()))
                if kind == "commitInfo":
                    print(f"    commitInfo  operation={body.get('operation')}  params={body.get('operationParameters')}")
                elif kind == "metaData":
                    cols = [fld["name"] for fld in json.loads(body["schemaString"])["fields"]]
                    print(f"    metaData    columns={cols}  partitionBy={body['partitionColumns']}")
                elif kind == "protocol":
                    print(f"    protocol    minReader={body['minReaderVersion']}  minWriter={body['minWriterVersion']}")
                elif kind == "add":
                    print(f"    add         {body['path']}  ({body['size']:,} bytes)")
                elif kind == "remove":
                    print(f"    remove      {body['path']}")
                elif kind == "txn":
                    print(f"    txn         appId={body['appId']}  version={body['version']}")
                else:
                    print(f"    {kind}")


def data_files(path):
    """Parquet files physically in the table folder (the log's own files excluded)."""
    files = glob.glob(f"{path}/**/*.parquet", recursive=True)
    files = [os.path.relpath(f, path).replace("\\", "/") for f in files]
    return sorted(f for f in files if not f.startswith("_delta_log"))


def attempt(fn):
    """Run fn(); on failure print the error type and its first line."""
    try:
        fn()
        print("succeeded")
    except Exception as e:
        first_line = str(e).strip().splitlines()[0][:300]
        print(f"FAILED -> {type(e).__name__}: {first_line}")

Our running example is a small table of online orders.

In [5]:
ORDER_SCHEMA = "order_id INT, customer STRING, product STRING, qty INT, price DOUBLE, status STRING"

orders = spark.createDataFrame([
    (1001, "alice", "laptop",     1, 1200.0, "shipped"),
    (1002, "bob",   "mouse",      2,   25.0, "shipped"),
    (1003, "carol", "monitor",    1,  300.0, "pending"),
    (1004, "dave",  "keyboard",   1,   80.0, "pending"),
    (1005, "alice", "headphones", 1,  150.0, "pending"),
    (1006, "erin",  "laptop",     1, 1100.0, "pending"),
    (1007, "frank", "tablet",     1,  600.0, "pending"),
    (1008, "bob",   "webcam",     1,   70.0, "shipped"),
], ORDER_SCHEMA).coalesce(2)   # 2 files keeps the logs short and readable

orders.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+--------+----------+---+------+-------+
|order_id|customer|   product|qty| price| status|
+--------+--------+----------+---+------+-------+
|    1001|   alice|    laptop|  1|1200.0|shipped|
|    1002|     bob|     mouse|  2|  25.0|shipped|
|    1003|   carol|   monitor|  1| 300.0|pending|
|    1004|    dave|  keyboard|  1|  80.0|pending|
|    1005|   alice|headphones|  1| 150.0|pending|
|    1006|    erin|    laptop|  1|1100.0|pending|
|    1007|   frank|    tablet|  1| 600.0|pending|
|    1008|     bob|    webcam|  1|  70.0|shipped|
+--------+--------+----------+---+------+-------+



---

### 1. The Problem: a Data Lake is Just a Folder of Files

First, the *plain Parquet* way. We'll break it in two ways that happen all the time in real pipelines.

#### 1a. Nothing enforces the schema

An upstream team renames `qty` to `quantity`. The append succeeds, and the table quietly becomes garbage.

In [6]:
PARQUET_PATH = f"{WORK_DIR}/orders_parquet"
orders.write.mode("overwrite").parquet(PARQUET_PATH)

bad_batch = spark.createDataFrame(
    [(1009, "gina", "phone", 1, 800.0, "pending")],
    "order_id INT, customer STRING, product STRING, quantity INT, price DOUBLE, status STRING")

attempt(lambda: bad_batch.write.mode("append").parquet(PARQUET_PATH))

spark.read.option("mergeSchema", "true").parquet(PARQUET_PATH).orderBy("order_id").show()

succeeded
+--------+--------+----------+----+------+-------+--------+
|order_id|customer|   product| qty| price| status|quantity|
+--------+--------+----------+----+------+-------+--------+
|    1001|   alice|    laptop|   1|1200.0|shipped|    NULL|
|    1002|     bob|     mouse|   2|  25.0|shipped|    NULL|
|    1003|   carol|   monitor|   1| 300.0|pending|    NULL|
|    1004|    dave|  keyboard|   1|  80.0|pending|    NULL|
|    1005|   alice|headphones|   1| 150.0|pending|    NULL|
|    1006|    erin|    laptop|   1|1100.0|pending|    NULL|
|    1007|   frank|    tablet|   1| 600.0|pending|    NULL|
|    1008|     bob|    webcam|   1|  70.0|shipped|    NULL|
|    1009|    gina|     phone|NULL| 800.0|pending|       1|
+--------+--------+----------+----+------+-------+--------+



**NOTE:**  
The table now has both `qty` and `quantity`, each containing null.  
Without `mergeSchema`, Spark picks the schema from whichever file it samples, so different readers can see different tables.

#### 1b. A failed overwrite destroys the data

A nightly job reprices every order and overwrites the table. Halfway through, a task crashes. We simulate the crash with a UDF (user-defined function) that throws on one row.

In [8]:
@F.udf("double")
def flaky_reprice(order_id, price):
    if order_id == 1007:
        raise ValueError("simulated crash in the middle of the job")
    return round(price * 1.10, 2)

repriced = orders.withColumn("price", flaky_reprice("order_id", "price"))

orders.write.mode("overwrite").parquet(PARQUET_PATH)       # start clean: 8 good rows
print("rows before:", spark.read.parquet(PARQUET_PATH).count())

attempt(lambda: repriced.write.mode("overwrite").parquet(PARQUET_PATH))

attempt(lambda: print("rows after: ", spark.read.parquet(PARQUET_PATH).count()))

rows before: 8


26/11/17 06:32:28 ERROR Utils: Aborting task
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/tmp/ipykernel_140492/482138079.py", line 4, in flaky_reprice
ValueError: simulated crash in the middle of the job
 SQLSTATE: 38000
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:665)
	at org.apache.spark.sql.execution.python.PythonArrowOutput$$anon$1.read(PythonArrowOutput.scala:142)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:602)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:611)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.

FAILED -> PythonException: An exception was thrown from the Python worker:
FAILED -> AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually. SQLSTATE: 42KD9


**NOTE:**  
Spark's Parquet writer deletes the old files *before* it writes the new ones. When the job dies, the old data is already gone and the new data never arrived. There is no undo. We'll repeat this exact experiment against a Delta table in section 3.

In [9]:
# try reading the data
spark.read.option("mergeSchema", "true").parquet(PARQUET_PATH).orderBy("order_id").show()

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually. SQLSTATE: 42KD9

---

### 2. Creating a Delta Table and Reading its Log

Writing a Delta table is one word different from writing Parquet: `.format("delta")`.

In [10]:
ORDERS_PATH = f"{WORK_DIR}/orders_delta"
orders.write.format("delta").mode("overwrite").save(ORDERS_PATH)

spark.read.format("delta").load(ORDERS_PATH).orderBy("order_id").show()

[Stage 30:=================================================>      (44 + 1) / 50]

+--------+--------+----------+---+------+-------+
|order_id|customer|   product|qty| price| status|
+--------+--------+----------+---+------+-------+
|    1001|   alice|    laptop|  1|1200.0|shipped|
|    1002|     bob|     mouse|  2|  25.0|shipped|
|    1003|   carol|   monitor|  1| 300.0|pending|
|    1004|    dave|  keyboard|  1|  80.0|pending|
|    1005|   alice|headphones|  1| 150.0|pending|
|    1006|    erin|    laptop|  1|1100.0|pending|
|    1007|   frank|    tablet|  1| 600.0|pending|
|    1008|     bob|    webcam|  1|  70.0|shipped|
+--------+--------+----------+---+------+-------+



We can also register the folder as a SQL table. The data stays where it is; the catalog just points at it.

In [11]:
spark.sql("DROP TABLE IF EXISTS orders")
spark.sql(f"CREATE TABLE orders USING DELTA LOCATION '{ORDERS_PATH}'")

spark.sql("SELECT status, COUNT(*) AS n, SUM(qty * price) AS revenue FROM orders GROUP BY status").show()

+-------+---+-------+
| status|  n|revenue|
+-------+---+-------+
|shipped|  3| 1320.0|
|pending|  5| 2230.0|
+-------+---+-------+



#### What's on disk?

In [12]:
print("orders_delta/")
for name in data_files(ORDERS_PATH):
    print("    " + name)
print("    _delta_log/")
for name in sorted(os.listdir(f"{ORDERS_PATH}/_delta_log")):
    if not name.endswith(".crc"):     # skip checksum files
        print("        " + name)

orders_delta/
    part-00000-e7506b9c-8a1e-485f-a3d5-285d3f84a968-c000.snappy.parquet
    _delta_log/
        00000000000000000000.json
        _staged_commits


Ordinary Parquet files, plus a `_delta_log/` folder holding `00000000000000000000.json`. That file **is** version 0 of the table. Each line in it is one **action**:

- `protocol`: which reader/writer features are needed to understand this table
- `metaData`: the schema and partition columns
- `add`: a data file that is part of the table from this version on
- `commitInfo`: who did what (operation, parameters, timestamp)

In [13]:
show_log(ORDERS_PATH)

--- version 0: _delta_log/00000000000000000000.json
    commitInfo  operation=WRITE  params={'mode': 'Overwrite', 'partitionBy': '[]'}
    metaData    columns=['order_id', 'customer', 'product', 'qty', 'price', 'status']  partitionBy=[]
    protocol    minReader=1  minWriter=2
    add         part-00000-e7506b9c-8a1e-485f-a3d5-285d3f84a968-c000.snappy.parquet  (2,036 bytes)


Here is one raw `add` action. Notice the `stats`: Delta records the row count and min/max of each column per file, so a query like `WHERE order_id = 1006` can skip files whose range can't contain it (**data skipping**). We talked about this earlier with predicate pushdown.

In [14]:
with open(f"{ORDERS_PATH}/_delta_log/00000000000000000000.json") as fh:
    add = next(json.loads(line)["add"] for line in fh if line.startswith('{"add"'))

add["stats"] = json.loads(add["stats"])
print(json.dumps(add, indent=2))

{
  "path": "part-00000-e7506b9c-8a1e-485f-a3d5-285d3f84a968-c000.snappy.parquet",
  "partitionValues": {},
  "size": 2036,
  "modificationTime": 1794915153938,
  "dataChange": true,
  "stats": {
    "numRecords": 8,
    "minValues": {
      "order_id": 1001,
      "customer": "alice",
      "product": "headphones",
      "qty": 1,
      "price": 25.0,
      "status": "pending"
    },
    "maxValues": {
      "order_id": 1008,
      "customer": "frank",
      "product": "webcam",
      "qty": 2,
      "price": 1200.0,
      "status": "shipped"
    },
    "nullCount": {
      "order_id": 0,
      "customer": 0,
      "product": 0,
      "qty": 0,
      "price": 0,
      "status": 0
    }
  }
}


**NOTE:**  
To read the table, Delta replays the log: start from nothing, applies each `add` and `remove` in order, and the files left over are the table. The folder listing is never trusted. This one rule is what makes everything in the rest of the notebook possible.

---

### 3. Atomicity: the Failed Overwrite, Again

Same crashing job as in 1b, but now the target is a Delta table.

In [15]:
print("rows before:", spark.read.format("delta").load(ORDERS_PATH).count())

attempt(lambda: repriced.write.format("delta").mode("overwrite").save(ORDERS_PATH))

print("rows after: ", spark.read.format("delta").load(ORDERS_PATH).count())
print("commits in log:", len(glob.glob(f"{ORDERS_PATH}/_delta_log/*.json")))

rows before: 8


26/11/17 06:33:53 ERROR Utils: Aborting task                        (0 + 1) / 1]
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/tmp/ipykernel_140492/482138079.py", line 4, in flaky_reprice
ValueError: simulated crash in the middle of the job
 SQLSTATE: 38000
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:665)
	at org.apache.spark.sql.execution.python.PythonArrowOutput$$anon$1.read(PythonArrowOutput.scala:142)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:602)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:611)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(U

FAILED -> Py4JJavaError: An error occurred while calling o206.save.
rows after:  8
commits in log: 1


**NOTE ON COMMIT:**  
The table is untouched and there is still only one commit. Delta writes new Parquet files first, and only after every task succeeds does it write the next `_delta_log/*.json` file. Creating that one file is the atomic commit. If the job dies before then, any half-written Parquet files are orphans that no reader will ever see.

**NOTE ON CONCURRENCY:**  
Two writers committing at once both try to create the same next log file (say `...01.json`). Only one can win; the loser checks whether the winner's changes conflict with its own and either retries on top of them or fails. This is **optimistic concurrency control**.

---

### 4. Schema Enforcement and Schema Evolution

#### Enforcement

The same `quantity` vs `qty` batch from 1a.

In [16]:
attempt(lambda: bad_batch.write.format("delta").mode("append").save(ORDERS_PATH))

FAILED -> AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.


**Rejected, and the table is unchanged.** Delta compares the incoming schema to the `metaData` action in the log before it commits.

#### Evolution

Sometimes the schema *should* change. Say orders now carry a `coupon` code. Opting in with `mergeSchema` adds the column in the same commit as the data. Existing rows read it as null.

In [17]:
new_orders = spark.createDataFrame([
    (1009, "gina",  "phone",   1, 800.0, "pending", "FALL10"),
    (1010, "henry", "charger", 3,  20.0, "pending", None),
], ORDER_SCHEMA + ", coupon STRING")

(new_orders.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(ORDERS_PATH))

spark.table("orders").orderBy("order_id").show()
show_log(ORDERS_PATH, versions=[1])

+--------+--------+----------+---+------+-------+------+
|order_id|customer|   product|qty| price| status|coupon|
+--------+--------+----------+---+------+-------+------+
|    1001|   alice|    laptop|  1|1200.0|shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|shipped|  NULL|
|    1003|   carol|   monitor|  1| 300.0|pending|  NULL|
|    1004|    dave|  keyboard|  1|  80.0|pending|  NULL|
|    1005|   alice|headphones|  1| 150.0|pending|  NULL|
|    1006|    erin|    laptop|  1|1100.0|pending|  NULL|
|    1007|   frank|    tablet|  1| 600.0|pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0|shipped|  NULL|
|    1009|    gina|     phone|  1| 800.0|pending|FALL10|
|    1010|   henry|   charger|  3|  20.0|pending|  NULL|
+--------+--------+----------+---+------+-------+------+

--- version 1: _delta_log/00000000000000000001.json
    commitInfo  operation=WRITE  params={'mode': 'Append', 'partitionBy': '[]', 'canMergeSchema': True}
    metaData    columns=['order_id', 'custome

**NOTE:**  
Version 1 contains a new `metaData` action with the wider schema. The old Parquet files were **not** rewritten; readers fill in null for columns a file doesn't have.

---

### 5. UPDATE and DELETE

Parquet files are immutable, so how can Delta update a row? It **rewrites the files that contain matching rows** and commits a `remove` for each old file plus an `add` for each new one.

We'll use the Python `DeltaTable` API and SQL side by side. They do exactly the same thing.

#### Update

In [18]:
orders_dt = DeltaTable.forPath(spark, ORDERS_PATH)

# Python API: carol's monitor ships
orders_dt.update(
    condition="order_id = 1003",
    set={"status": "'shipped'"})

# SQL: dave cancels
spark.sql("UPDATE orders SET status = 'cancelled' WHERE order_id = 1004")

spark.table("orders").orderBy("order_id").show()

[Stage 89:=========================================>              (37 + 1) / 50]

+--------+--------+----------+---+------+---------+------+
|order_id|customer|   product|qty| price|   status|coupon|
+--------+--------+----------+---+------+---------+------+
|    1001|   alice|    laptop|  1|1200.0|  shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|  shipped|  NULL|
|    1003|   carol|   monitor|  1| 300.0|  shipped|  NULL|
|    1004|    dave|  keyboard|  1|  80.0|cancelled|  NULL|
|    1005|   alice|headphones|  1| 150.0|  pending|  NULL|
|    1006|    erin|    laptop|  1|1100.0|  pending|  NULL|
|    1007|   frank|    tablet|  1| 600.0|  pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0|  shipped|  NULL|
|    1009|    gina|     phone|  1| 800.0|  pending|FALL10|
|    1010|   henry|   charger|  3|  20.0|  pending|  NULL|
+--------+--------+----------+---+------+---------+------+



#### Conditional update

Business rule: pending orders over $500 get a 10% discount. The condition and the new value can both be expressions over the row.

In [19]:
orders_dt.update(
    condition=(F.col("status") == "pending") & (F.col("price") > 500),
    set={"price": F.round(F.col("price") * 0.9, 2)})

spark.table("orders").orderBy("order_id").show()

[Stage 106:===============================================>       (43 + 1) / 50]

+--------+--------+----------+---+------+---------+------+
|order_id|customer|   product|qty| price|   status|coupon|
+--------+--------+----------+---+------+---------+------+
|    1001|   alice|    laptop|  1|1200.0|  shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|  shipped|  NULL|
|    1003|   carol|   monitor|  1| 300.0|  shipped|  NULL|
|    1004|    dave|  keyboard|  1|  80.0|cancelled|  NULL|
|    1005|   alice|headphones|  1| 150.0|  pending|  NULL|
|    1006|    erin|    laptop|  1| 990.0|  pending|  NULL|
|    1007|   frank|    tablet|  1| 540.0|  pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0|  shipped|  NULL|
|    1009|    gina|     phone|  1| 720.0|  pending|FALL10|
|    1010|   henry|   charger|  3|  20.0|  pending|  NULL|
+--------+--------+----------+---+------+---------+------+



#### Delete

In [20]:
orders_dt.delete("status = 'cancelled'")
# equivalent SQL: DELETE FROM orders WHERE status = 'cancelled'

spark.table("orders").orderBy("order_id").show()

[Stage 123:=====================================>                 (34 + 1) / 50]

+--------+--------+----------+---+------+-------+------+
|order_id|customer|   product|qty| price| status|coupon|
+--------+--------+----------+---+------+-------+------+
|    1001|   alice|    laptop|  1|1200.0|shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|shipped|  NULL|
|    1003|   carol|   monitor|  1| 300.0|shipped|  NULL|
|    1005|   alice|headphones|  1| 150.0|pending|  NULL|
|    1006|    erin|    laptop|  1| 990.0|pending|  NULL|
|    1007|   frank|    tablet|  1| 540.0|pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0|shipped|  NULL|
|    1009|    gina|     phone|  1| 720.0|pending|FALL10|
|    1010|   henry|   charger|  3|  20.0|pending|  NULL|
+--------+--------+----------+---+------+-------+------+



In [21]:
show_log(ORDERS_PATH, versions=[2, 3, 4, 5])

--- version 2: _delta_log/00000000000000000002.json
    commitInfo  operation=UPDATE  params={'predicate': '["(order_id#2933 = 1003)"]'}
    add         part-00000-a4c4d690-d7b8-430c-9854-3cbd88bfcc88-c000.snappy.parquet  (2,241 bytes)
    remove      part-00000-e7506b9c-8a1e-485f-a3d5-285d3f84a968-c000.snappy.parquet
--- version 3: _delta_log/00000000000000000003.json
    commitInfo  operation=UPDATE  params={'predicate': '["(order_id#3613 = 1004)"]'}
    add         part-00000-aa55806b-2f1a-450f-86c7-c18f37d8dde2-c000.snappy.parquet  (2,259 bytes)
    remove      part-00000-a4c4d690-d7b8-430c-9854-3cbd88bfcc88-c000.snappy.parquet
--- version 4: _delta_log/00000000000000000004.json
    commitInfo  operation=UPDATE  params={'predicate': '["((status#2938 = pending) AND (price#2937 > 500.0))"]'}
    add         part-00000-2291cc67-4804-4112-9a9a-686cb38856dd-c000.snappy.parquet  (2,347 bytes)
    remove      part-00000-3c0f0c57-a6a1-4130-a8d3-0cdfb1aba637-c000.snappy.parquet
    remove  

**NOTE:**  
Every change is a `remove` of old files plus an `add` of rewritten ones. The removed files are *still on disk*. Watch what happens if we bypass the log and read the folder as plain Parquet:

In [22]:
print("files on disk:         ", len(data_files(ORDERS_PATH)))
print("files in current table:", spark.sql("DESCRIBE DETAIL orders").first()["numFiles"])
print()
print("rows via Delta:        ", spark.read.format("delta").load(ORDERS_PATH).count())
print("rows via plain Parquet:", spark.read.option("mergeSchema", "true").parquet(ORDERS_PATH).count())

files on disk:          7
files in current table: 1



rows via Delta:         9
rows via plain Parquet: 45


The Parquet reader sees every version of every row at once. Always read a Delta table through Delta. Those old files make time travel work (section 7).

---

### 6. MERGE (Upsert)

A common pipeline step: a batch of changes arrives from an operational system. Some rows are updates to existing orders, some are brand new orders. `MERGE` applies the whole batch in **one atomic commit**:

- matched and the status changed: update it
- matched and the order was returned: delete it
- not matched: insert it

In [24]:
changes = spark.createDataFrame([
    (1005, "alice", "headphones", 1,  150.0, "shipped",  None),      # update
    (1006, "erin",  "laptop",     1,  990.0, "shipped",  None),      # update
    (1002, "bob",   "mouse",      2,   25.0, "returned", None),      # delete
    (1008, "bob",   "webcam",     1,   70.0, "shipped",  None),      # no change: skipped
    (1011, "ivy",   "laptop",     1, 1250.0, "pending",  "VIP"),     # insert
    (1012, "jack",  "mouse",      1,   25.0, "pending",  None),      # insert
], ORDER_SCHEMA + ", coupon STRING")

(orders_dt.alias("t")
    .merge(changes.alias("s"), "t.order_id = s.order_id")
    .whenMatchedDelete(condition="s.status = 'returned'")
    .whenMatchedUpdate(condition="s.status <> t.status",
                       set={"status": "s.status"})
    .whenNotMatchedInsertAll()
    .execute())

spark.table("orders").orderBy("order_id").show()

[Stage 184:========================================>              (37 + 1) / 50]

+--------+--------+----------+---+------+--------+------+
|order_id|customer|   product|qty| price|  status|coupon|
+--------+--------+----------+---+------+--------+------+
|    1001|   alice|    laptop|  1|1200.0| shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|returned|  NULL|
|    1003|   carol|   monitor|  1| 300.0| shipped|  NULL|
|    1005|   alice|headphones|  1| 150.0| shipped|  NULL|
|    1006|    erin|    laptop|  1| 990.0| shipped|  NULL|
|    1007|   frank|    tablet|  1| 540.0| pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0| shipped|  NULL|
|    1009|    gina|     phone|  1| 720.0| pending|FALL10|
|    1010|   henry|   charger|  3|  20.0| pending|  NULL|
|    1011|     ivy|    laptop|  1|1250.0| pending|   VIP|
|    1012|    jack|     mouse|  1|  25.0| pending|  NULL|
+--------+--------+----------+---+------+--------+------+



The same merge in SQL looks like this (not run, since the changes are already applied):

```sql
MERGE INTO orders AS t
USING changes AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.status = 'returned' THEN DELETE
WHEN MATCHED AND s.status <> t.status THEN UPDATE SET status = s.status
WHEN NOT MATCHED THEN INSERT *
```

`DESCRIBE HISTORY` reports what the merge did in its `operationMetrics`:

In [25]:
metrics = orders_dt.history(1).first()["operationMetrics"]
for k in ["numSourceRows", "numTargetRowsInserted", "numTargetRowsUpdated",
          "numTargetRowsDeleted", "numTargetFilesAdded", "numTargetFilesRemoved"]:
    print(f"{k:24s} {metrics.get(k)}")

numSourceRows            6
numTargetRowsInserted    1
numTargetRowsUpdated     0
numTargetRowsDeleted     0
numTargetFilesAdded      1
numTargetFilesRemoved    1


**NOTE:**  
Before Delta, an upsert on a data lake meant reading the whole table, joining it with the changes, and overwriting everything (and hoping nothing crashed, see 1b). `MERGE` rewrites only the files that contain matched rows.

---

### 7. Time Travel

Every version of the table is still described by the log, and its files are still on disk. So we can read any of them.

In [26]:
(orders_dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
    .show(truncate=60))

+-------+-----------------------+---------+------------------------------------------------------------+
|version|              timestamp|operation|                                         operationParameters|
+-------+-----------------------+---------+------------------------------------------------------------+
|      0|2026-11-17 06:32:36.095|    WRITE|                      {mode -> Overwrite, partitionBy -> []}|
|      1|2026-11-17 06:39:45.178|    WRITE| {mode -> Append, partitionBy -> [], canMergeSchema -> true}|
|      2|2026-11-17 06:43:29.045|   UPDATE|                   {predicate -> ["(order_id#2933 = 1003)"]}|
|      3|2026-11-17 06:43:42.177|   UPDATE|                   {predicate -> ["(order_id#3613 = 1004)"]}|
|      4|2026-11-17 06:45:53.739|   UPDATE|{predicate -> ["((status#2938 = pending) AND (price#2937 ...|
|      5|2026-11-17 06:48:55.272|   DELETE|                {predicate -> ["(status#2938 = cancelled)"]}|
|      6| 2026-11-17 06:57:45.87|    MERGE|{predicate -

Read by **version number** (Python or SQL):

In [28]:
print("version 0, the original 8 orders:")
spark.read.format("delta").option("versionAsOf", 0).load(ORDERS_PATH).orderBy("order_id").show()

print("version 3, after the first two updates:")
spark.sql("SELECT order_id, customer, status FROM orders VERSION AS OF 3 ORDER BY order_id").show()

version 0, the original 8 orders:


+--------+--------+----------+---+------+-------+
|order_id|customer|   product|qty| price| status|
+--------+--------+----------+---+------+-------+
|    1001|   alice|    laptop|  1|1200.0|shipped|
|    1002|     bob|     mouse|  2|  25.0|shipped|
|    1003|   carol|   monitor|  1| 300.0|pending|
|    1004|    dave|  keyboard|  1|  80.0|pending|
|    1005|   alice|headphones|  1| 150.0|pending|
|    1006|    erin|    laptop|  1|1100.0|pending|
|    1007|   frank|    tablet|  1| 600.0|pending|
|    1008|     bob|    webcam|  1|  70.0|shipped|
+--------+--------+----------+---+------+-------+

version 3, after the first two updates:


[Stage 226:=============================================>         (41 + 1) / 50]

+--------+--------+---------+
|order_id|customer|   status|
+--------+--------+---------+
|    1001|   alice|  shipped|
|    1002|     bob|  shipped|
|    1003|   carol|  shipped|
|    1004|    dave|cancelled|
|    1005|   alice|  pending|
|    1006|    erin|  pending|
|    1007|   frank|  pending|
|    1008|     bob|  shipped|
|    1009|    gina|  pending|
|    1010|   henry|  pending|
+--------+--------+---------+



Or by **timestamp**. Delta picks the latest version committed at or before that time.

In [30]:
ts_v1 = orders_dt.history().filter("version = 1").first()["timestamp"]
print("version 1 was committed at", ts_v1)

(spark.read.format("delta")
    .option("timestampAsOf", ts_v1.strftime("%Y-%m-%d %H:%M:%S.%f"))
    .load(ORDERS_PATH)
    .orderBy("order_id")
    .show())

version 1 was committed at 2026-11-17 06:39:45.178000


[Stage 239:=====================================>                 (34 + 1) / 50]

+--------+--------+----------+---+------+-------+------+
|order_id|customer|   product|qty| price| status|coupon|
+--------+--------+----------+---+------+-------+------+
|    1001|   alice|    laptop|  1|1200.0|shipped|  NULL|
|    1002|     bob|     mouse|  2|  25.0|shipped|  NULL|
|    1003|   carol|   monitor|  1| 300.0|pending|  NULL|
|    1004|    dave|  keyboard|  1|  80.0|pending|  NULL|
|    1005|   alice|headphones|  1| 150.0|pending|  NULL|
|    1006|    erin|    laptop|  1|1100.0|pending|  NULL|
|    1007|   frank|    tablet|  1| 600.0|pending|  NULL|
|    1008|     bob|    webcam|  1|  70.0|shipped|  NULL|
|    1009|    gina|     phone|  1| 800.0|pending|FALL10|
|    1010|   henry|   charger|  3|  20.0|pending|  NULL|
+--------+--------+----------+---+------+-------+------+



#### Use case: audit what changed

Diff two versions with ordinary DataFrame operations.

In [31]:
cols = ["order_id", "customer", "product", "price", "status"]
v0 = spark.read.format("delta").option("versionAsOf", 0).load(ORDERS_PATH).select(cols)
now = spark.table("orders").select(cols)

print("rows in version 0 that are gone or changed now:")
v0.exceptAll(now).orderBy("order_id").show()

print("rows now that are new or changed since version 0:")
now.exceptAll(v0).orderBy("order_id").show()

rows in version 0 that are gone or changed now:


+--------+--------+----------+------+-------+
|order_id|customer|   product| price| status|
+--------+--------+----------+------+-------+
|    1002|     bob|     mouse|  25.0|shipped|
|    1003|   carol|   monitor| 300.0|pending|
|    1004|    dave|  keyboard|  80.0|pending|
|    1005|   alice|headphones| 150.0|pending|
|    1006|    erin|    laptop|1100.0|pending|
|    1007|   frank|    tablet| 600.0|pending|
+--------+--------+----------+------+-------+

rows now that are new or changed since version 0:


+--------+--------+----------+------+--------+
|order_id|customer|   product| price|  status|
+--------+--------+----------+------+--------+
|    1002|     bob|     mouse|  25.0|returned|
|    1003|   carol|   monitor| 300.0| shipped|
|    1005|   alice|headphones| 150.0| shipped|
|    1006|    erin|    laptop| 990.0| shipped|
|    1007|   frank|    tablet| 540.0| pending|
|    1009|    gina|     phone| 720.0| pending|
|    1010|   henry|   charger|  20.0| pending|
|    1011|     ivy|    laptop|1250.0| pending|
|    1012|    jack|     mouse|  25.0| pending|
+--------+--------+----------+------+--------+



#### Use case: roll back a bad job

Someone runs a `DELETE` with the wrong condition and wipes out every pending order.

In [33]:
good_version = orders_dt.history(1).first()["version"]

spark.sql("DELETE FROM orders WHERE status = 'pending'")     # oops
print("after the bad delete:", spark.table("orders").count(), "rows")

orders_dt.restoreToVersion(good_version)
# equivalent SQL: RESTORE TABLE orders TO VERSION AS OF <good_version>
print(f"after restoring to version {good_version}:", spark.table("orders").count(), "rows")

orders_dt.history(3).select("version", "operation", "operationParameters").show(truncate=False)

after the bad delete: 6 rows


after restoring to version 9: 11 rows
+-------+---------+-------------------------------------------+
|version|operation|operationParameters                        |
+-------+---------+-------------------------------------------+
|11     |RESTORE  |{version -> 9, timestamp -> NULL}          |
|10     |DELETE   |{predicate -> ["(status#22354 = pending)"]}|
|9      |RESTORE  |{version -> 7, timestamp -> NULL}          |
+-------+---------+-------------------------------------------+



**NOTE:**  
`RESTORE` doesn't erase history. It's a *new* commit that re-adds the old files and removes the current ones, so the bad delete is still in the audit trail.

---

### 8. Partitioning, OPTIMIZE, and VACUUM

Streaming and frequent small batch jobs create lots of **small files**, and every file costs a task, an open, and a footer read. Let's make a click-log table partitioned by `country`, written in 10 tiny appends.

In [ ]:
CLICKS_PATH = f"{WORK_DIR}/clicks"

def click_batch(i, n=300):
    return (spark.range(i * n, (i + 1) * n)
        .select(
            F.col("id").alias("click_id"),
            (F.rand(seed=i) * 1000).cast("int").alias("user_id"),
            F.element_at(F.array(F.lit("US"), F.lit("DE"), F.lit("IN")),
                         (F.col("id") % 3 + 1).cast("int")).alias("country"),
            F.element_at(F.array(F.lit("/home"), F.lit("/search"), F.lit("/cart"), F.lit("/checkout")),
                         (F.rand(seed=100 + i) * 4 + 1).cast("int")).alias("page")))

for i in range(10):
    (click_batch(i).coalesce(1)
        .write.format("delta")
        .mode("append")
        .partitionBy("country")
        .save(CLICKS_PATH))

clicks_dt = DeltaTable.forPath(spark, CLICKS_PATH)

def file_report(label):
    n_active = clicks_dt.detail().first()["numFiles"]
    print(f"{label:22s} files in table: {n_active:3d}   files on disk: {len(data_files(CLICKS_PATH)):3d}")

file_report("after 10 appends")
print(sorted(os.listdir(CLICKS_PATH)))

Partitioning puts each `country` in its own `country=XX/` folder, so `WHERE country = 'DE'` reads only that folder. 10 appends x 3 partitions = 30 small files.

#### OPTIMIZE: compact small files

`OPTIMIZE` rewrites the small files in each partition into larger ones. Adding **Z-ordering** sorts the data by the given column(s) while compacting, which makes the per-file min/max stats tight, so data skipping on `user_id` gets much better.

In [ ]:
clicks_dt.optimize().executeZOrderBy("user_id")
# plain compaction:  clicks_dt.optimize().executeCompaction()
# SQL:               OPTIMIZE delta.`<path>` ZORDER BY (user_id)

file_report("after OPTIMIZE")

**Point out:** the table now has 3 files (one per partition), but all 33 are still on disk. `OPTIMIZE` is just another commit (30 `remove`s, 3 `add`s), so time travel to the pre-optimize versions still works. The data itself didn't change, only its layout.

That `OPTIMIZE` was version 10, so look in `_delta_log/` too. Every 10 commits Delta writes a **checkpoint** (`...10.checkpoint.parquet`): a snapshot of the whole table state, so readers start there instead of replaying every JSON file from version 0.

In [ ]:
sorted(f for f in os.listdir(f"{CLICKS_PATH}/_delta_log") if not f.endswith(".crc"))

#### VACUUM: delete files no longer referenced

`VACUUM` physically deletes files that were removed from the table more than the retention period ago (default **7 days**). We use 0 hours for the demo, which we allowed in the setup cell.

In [ ]:
clicks_dt.vacuum(0)
file_report("after VACUUM")

print("\ntime travel to version 0 after VACUUM:")
attempt(lambda: spark.read.format("delta").option("versionAsOf", 0).load(CLICKS_PATH).count())

**NOTE:**  
`VACUUM` is the one operation that limits time travel. The log still lists version 0, but its files are gone. The retention period is a trade-off between storage cost and how far back you can travel. It also protects readers and writers that are still using older files, which is why a 0-hour retention is unsafe on a live table.

---

### 9. Delta Tables in a Streaming Pipeline

A Delta table can be a Structured Streaming **source** (new commits are new data) and **sink**. A common layout is the *medallion* pattern: a raw (bronze) table, a cleaned (silver) table, and aggregates (gold).

We stream raw sensor readings into a cleaned table. `trigger(availableNow=True)` processes whatever is new and then stops, which is handy for scheduled jobs (and notebooks).

In [ ]:
RAW_PATH = f"{WORK_DIR}/readings_bronze"
CLEAN_PATH = f"{WORK_DIR}/readings_silver"
CHECKPOINT = f"{WORK_DIR}/_checkpoints/readings_silver"

def land_raw(start, n):
    """Append n raw readings; some are out of range (sensor glitches)."""
    (spark.range(start, start + n)
        .select(
            F.col("id").alias("reading_id"),
            (F.col("id") % 5).cast("int").alias("sensor_id"),
            F.round(F.rand(seed=start) * 140 - 10, 1).alias("temp_f"),
            F.current_timestamp().alias("ingested_at"))
        .write.format("delta").mode("append").save(RAW_PATH))

def run_pipeline():
    """Bronze -> silver: drop glitches, add Celsius. Processes only new commits."""
    query = (spark.readStream.format("delta").load(RAW_PATH)
        .filter("temp_f BETWEEN 0 AND 120")
        .withColumn("temp_c", F.round((F.col("temp_f") - 32) * 5 / 9, 1))
        .writeStream.format("delta")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start(CLEAN_PATH))
    query.awaitTermination()
    raw = spark.read.format("delta").load(RAW_PATH).count()
    clean = spark.read.format("delta").load(CLEAN_PATH).count()
    print(f"bronze rows: {raw:4d}   silver rows: {clean:4d}")

land_raw(0, 200)
run_pipeline()

More data lands. Running the pipeline again picks up **only** the new commit. Running it a third time with nothing new does nothing.

In [ ]:
land_raw(200, 100)
run_pipeline()

run_pipeline()     # nothing new: silver is unchanged

In [ ]:
show_log(CLEAN_PATH)

**NOTE:**  
Each streaming commit includes a `txn` action holding the query's id and its batch number. If the query crashes after writing files but before updating its checkpoint, it reruns the batch on restart; Delta sees that batch number is already in the log and skips the duplicate write. That is how the Delta sink gives **exactly-once** results, the problem we wrestled with in the Kafka and streaming modules.

One caveat: by default a Delta streaming source expects an **append-only** table. If the source table has `UPDATE`s or `DELETE`s, the stream fails unless you set `skipChangeCommits` or read the [Change Data Feed](https://docs.delta.io/latest/delta-change-data-feed.html) instead.

---

### SUMMARY

| Feature | How the log makes it work |
|---|---|
| Atomic writes | data files first, then one new `_delta_log/N.json` file = commit |
| Concurrent writers | only one writer can create `N.json`; others retry or fail |
| Schema enforcement / evolution | the `metaData` action holds the schema; writes are checked against it |
| UPDATE / DELETE / MERGE | rewrite affected files; commit `remove` + `add` |
| Time travel / RESTORE | replay the log only up to version N |
| Data skipping | per-file min/max `stats` in each `add` action |
| OPTIMIZE | a commit that swaps many small files for a few large ones |
| VACUUM | deletes files that no recent version references |
| Exactly-once streaming | `txn` action records which batches were committed |

---

### TRY IT

1. Open `delta_workspace/orders_delta/_delta_log/` in a text editor and find the commit for the `MERGE`. Match each `add` and `remove` to the rows that changed.
2. Run a second `restoreToVersion(0)` on `orders`. What does the `coupon` column look like afterward? Why?
3. Write the `clicks` table again without `partitionBy`, run `OPTIMIZE` with and without `ZORDER BY (user_id)`, and compare the `minValues`/`maxValues` of `user_id` in the new `add` actions. Which layout lets `WHERE user_id = 42` skip more files?
4. Enable the change data feed on a new table with `.option("delta.enableChangeDataFeed", "true")`, run some updates and deletes, then read the changes with `spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 0)`. What do the `_change_type` values mean?
5. In section 9, add a gold table that streams the silver table into per-sensor average temperatures. Which `outputMode` do you need, and why?

In [ ]:
spark.stop()